<a href="https://colab.research.google.com/github/Aswathi-06/Plant_Species_Classification/blob/main/Plant_species_classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
import tensorflow as tf
print("GPU available:", tf.config.list_physical_devices('GPU'))

GPU available: []


Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


create the project folder

In [ ]:
import os
project_path = "/content/drive/MyDrive/Plant_Identification_Project"
os.makedirs(project_path, exist_ok=True)
print("Project folder ready at:", project_path)

Project folder ready at: /content/drive/MyDrive/Plant_Identification_Project


Kaggle authentication

In [ ]:
from getpass import getpass
os.environ["KAGGLE_API_TOKEN"] = getpass("Paste your Kaggle API token here: ")

Paste your Kaggle API token here: ··········


Download and unzip

In [ ]:
!pip install -q kaggle
%cd /content
!kaggle datasets download -d abdallahalidev/plantvillage-dataset
!unzip -q plantvillage-dataset.zip -d plant_data_full

/content
Dataset URL: https://www.kaggle.com/datasets/abdallahalidev/plantvillage-dataset
License(s): CC-BY-NC-SA-4.0
100% 2.04G/2.04G [00:59<00:00, 37.1MB/s]



Verify the local extraction is complete BEFORE copying anywhere

In [ ]:
local_color_path = "/content/plant_data_full/plantvillage dataset/color"
local_classes = sorted(os.listdir(local_color_path))
print(f"Local extraction: {len(local_classes)} classes")

Local extraction: 38 classes


Copy only the color folder to Drive

In [12]:
import shutil

shutil.rmtree(destination, ignore_errors=True)
print("Old folder removed.")

# Confirm it's really gone before copying
print("Still exists:", os.path.exists(destination))

Old folder removed.
Still exists: False


In [13]:
import shutil

destination = "/content/drive/MyDrive/Plant_Identification_Project/plant_data/color"
shutil.copytree(local_color_path, destination)
print("Copy finished running.")

Copy finished running.


In [14]:
local_classes = sorted(os.listdir(local_color_path))
drive_classes = sorted(os.listdir(destination))
print(f"Drive copy: {len(drive_classes)} classes")

missing = set(local_classes) - set(drive_classes)
if missing:
    print(f"⚠️ INCOMPLETE — missing: {missing}")
else:
    print("✅ Confirmed complete — all 38 classes present.")

Drive copy: 38 classes
✅ Confirmed complete — all 38 classes present.


#From here rerun 3 cells

Build the full file list, grouped by species

In [15]:
import os

data_dir = "/content/drive/MyDrive/Plant_Identification_Project/plant_data/color"
classes = sorted(os.listdir(data_dir))

def get_species_from_folder(folder_name):
    return folder_name.split("___")[0]

# Build a dictionary: species -> list of full image file paths
species_files = {}

for folder in classes:
    species = get_species_from_folder(folder)
    folder_path = os.path.join(data_dir, folder)
    image_files = [os.path.join(folder_path, f) for f in os.listdir(folder_path)]

    if species not in species_files:
        species_files[species] = []
    species_files[species].extend(image_files)

# Quick check
for species, files in species_files.items():
    print(f"{species}: {len(files)} images")

Apple: 3171 images
Blueberry: 1502 images
Cherry_(including_sour): 1906 images
Corn_(maize): 3852 images
Grape: 4062 images
Orange: 5507 images
Peach: 2657 images
Pepper,_bell: 2475 images
Potato: 2152 images
Raspberry: 371 images
Soybean: 5090 images
Squash: 1835 images
Strawberry: 1565 images
Tomato: 18160 images


Stratified split into train/val/test (70/15/15)

In [16]:
from sklearn.model_selection import train_test_split

train_files = {}
val_files = {}
test_files = {}

for species, files in species_files.items():
    # First split: 70% train, 30% temp (which we'll split again into val/test)
    train, temp = train_test_split(files, train_size=0.7, random_state=42)
    # Second split: divide the remaining 30% evenly into val (15%) and test (15%)
    val, test = train_test_split(temp, train_size=0.5, random_state=42)

    train_files[species] = train
    val_files[species] = val
    test_files[species] = test

    print(f"{species}: train={len(train)}, val={len(val)}, test={len(test)}")

Apple: train=2219, val=476, test=476
Blueberry: train=1051, val=225, test=226
Cherry_(including_sour): train=1334, val=286, test=286
Corn_(maize): train=2696, val=578, test=578
Grape: train=2843, val=609, test=610
Orange: train=3854, val=826, test=827
Peach: train=1859, val=399, test=399
Pepper,_bell: train=1732, val=371, test=372
Potato: train=1506, val=323, test=323
Raspberry: train=259, val=56, test=56
Soybean: train=3563, val=763, test=764
Squash: train=1284, val=275, test=276
Strawberry: train=1095, val=235, test=235
Tomato: train=12712, val=2724, test=2724


Physically copy files into the new folder structure (locally)

In [17]:
import shutil
from tqdm import tqdm  # progress bar, makes long copies easier to monitor

local_split_dir = "/content/plant_data_split"

def copy_files(file_dict, split_name):
    for species, files in file_dict.items():
        dest_folder = os.path.join(local_split_dir, split_name, species)
        os.makedirs(dest_folder, exist_ok=True)
        for src_path in tqdm(files, desc=f"{split_name}/{species}", leave=False):
            filename = os.path.basename(src_path)
            shutil.copy(src_path, os.path.join(dest_folder, filename))

copy_files(train_files, "train")
copy_files(val_files, "val")
copy_files(test_files, "test")

print("Done copying all splits.")

Done copying all splits.


Verify the final structure

In [18]:
for split in ["train", "val", "test"]:
    split_path = os.path.join(local_split_dir, split)
    species_list = sorted(os.listdir(split_path))
    total = sum(len(os.listdir(os.path.join(split_path, s))) for s in species_list)
    print(f"{split}: {len(species_list)} species, {total} total images")

train: 14 species, 38007 total images
val: 14 species, 8146 total images
test: 14 species, 8152 total images


Resizing to 224×224

In [19]:
import tensorflow as tf

IMG_SIZE = (224, 224)
BATCH_SIZE = 32

train_dir = "/content/plant_data_split/train"
val_dir = "/content/plant_data_split/val"
test_dir = "/content/plant_data_split/test"

train_ds = tf.keras.utils.image_dataset_from_directory(
    train_dir,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="categorical",
    shuffle=True,
    seed=42
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    val_dir,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="categorical",
    shuffle=False
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    test_dir,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="categorical",
    shuffle=False
)

# Save the class names in the order Keras assigned them - we'll need this later
class_names = train_ds.class_names
print("Class order:", class_names)

Found 38007 files belonging to 14 classes.
Found 8146 files belonging to 14 classes.
Found 8152 files belonging to 14 classes.
Class order: ['Apple', 'Blueberry', 'Cherry_(including_sour)', 'Corn_(maize)', 'Grape', 'Orange', 'Peach', 'Pepper,_bell', 'Potato', 'Raspberry', 'Soybean', 'Squash', 'Strawberry', 'Tomato']


pixel normalization

In [20]:
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

def preprocess(image, label):
    image = preprocess_input(image)
    return image, label

train_ds = train_ds.map(preprocess)
val_ds = val_ds.map(preprocess)
test_ds = test_ds.map(preprocess)

# Improves training speed by preparing future batches while current one trains
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.prefetch(buffer_size=AUTOTUNE)
test_ds = test_ds.prefetch(buffer_size=AUTOTUNE)